In [ ]:
def download_and_extract(slide_id, target_dir="/content", force_redownload=False):
    """
    Downloads and extracts the STHELAR file with robust error checking.
    """
    # Updated URL with the correct path provided by the user
    url = f"https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/146/S-BIAD2146/Files/STHELAR/sdata_slides/sdata_{slide_id}.zarr.zip"
    local_zip = os.path.join(target_dir, f"{slide_id}.zip")
    extract_path = os.path.join(target_dir, f"sdata_{slide_id}")

    # 1. Cleanup if forcing a redownload
    if force_redownload and os.path.exists(local_zip):
        print(f"Removing existing file: {local_zip}")
        os.remove(local_zip)

    # 2. Download with HTTP check
    if not os.path.exists(local_zip):
        print(f"Downloading {slide_id} from {url}...")
        try:
            response = requests.get(url, stream=True, timeout=30)
            if response.status_code != 200:
                print(f"Error: Server returned status {response.status_code}")
                print("The link might be temporarily down or the slide ID is incorrect.")
                return None

            total_size = int(response.headers.get('content-length', 0))
            with open(local_zip, 'wb') as f, tqdm(total=total_size, unit='iB', unit_scale=True, desc=f"{slide_id}.zip") as bar:
                for data in response.iter_content(chunk_size=1024*1024):
                    size = f.write(data)
                    bar.update(size)
        except Exception as e:
            print(f"Download failed: {e}")
            if os.path.exists(local_zip): os.remove(local_zip)
            return None

    # 3. Safe Extraction
    print("Verifying and extracting zip file...")
    try:
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            # Check for corruption before extracting
            if zip_ref.testzip() is not None:
                raise zipfile.BadZipFile("CRC check failed. File is corrupted.")
            zip_ref.extractall(extract_path)
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip archive.")
        print("Deleting corrupted file. Please try running with force_redownload=True.")
        os.remove(local_zip)
        return None

    print(f"Ready! Data extracted to: {extract_path}")
    return extract_path

def process_stelar_slide(slide_id, input_dir="/content", output_dir="/content/drive/MyDrive/NicheMRF"):
    """
    Finds the correctly nested .zarr, subsets it, and saves to Drive.
    """
    # Look for the nested folder pattern seen in BioStudies
    potential_path = os.path.join(input_dir, f"sdata_{slide_id}", f"sdata_{slide_id}.zarr")
    if not os.path.exists(potential_path):
        potential_path = os.path.join(input_dir, f"sdata_{slide_id}.zarr")

    if not os.path.exists(potential_path):
        print(f"Error: Could not find .zarr folder at {potential_path}")
        return None

    print(f"Reading SpatialData from: {potential_path}")
    sdata = sd.read_zarr(potential_path)
    adata = sdata.tables["table_cells"]

    # Pre-processing: Filter out 'Less10' and take a 50k pilot subset
    adata = adata[adata.obs['final_label'] != 'Less10'].copy()
    adata_subset = adata[:50000].copy()

    # Save to Drive
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{slide_id}_pilot.h5ad")
    adata_subset.write(file_path)

    print(f"Success! Subset saved to: {file_path}")
    return file_path

def download_to_local_machine(file_path):
    """Triggers browser download from Colab."""
    from google.colab import files
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print("File not found for download.")

def load_sdata_object(slide_id, input_dir="/content"):
    """
    Loads the SpatialData object from the extracted .zarr directory.
    """
    potential_path = os.path.join(input_dir, f"sdata_{slide_id}", f"sdata_{slide_id}.zarr")
    if not os.path.exists(potential_path):
        potential_path = os.path.join(input_dir, f"sdata_{slide_id}.zarr")

    if not os.path.exists(potential_path):
        print(f"Error: Could not find .zarr folder at {potential_path}")
        return None

    print(f"Reading SpatialData from: {potential_path}")
    sdata = sd.read_zarr(potential_path)
    return sdata

In [ ]:
!pip install spatialdata scanpy requests tqdm squidpy
import spatialdata as sd
import scanpy as sc
import os
import zipfile
import requests
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.3 MB/s eta 0:00:00
   ━━━━

In [ ]:
slide_id = "lymph_node_s0"
path = download_and_extract(slide_id, force_redownload=True)
lite_path = None
if path:
    lite_path = process_stelar_slide(slide_id)
if lite_path:
    download_to_local_machine(lite_path)

lymph_node_s0.zip: 100%|██████████| 26.6G/26.6G [13:56<00:00, 31.8MiB/s]


Verifying and extracting zip file...
Ready! Data extracted to: /content/sdata_lymph_node_s0
Reading SpatialData from: /content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr


/tmp/ipykernel_1283/2951674298.py:66: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(potential_path)


In [ ]:
import os
from pathlib import Path

data_dir = Path("/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr")


# Check what's available
print("Cell boundaries:", list(data_dir.glob("shapes/cell_boundaries/*"))[:5])
print("HE patches:", list(data_dir.glob("shapes/he_patches/*"))[:5])
print("Tables:", list(data_dir.glob("tables/*")))

Cell boundaries: [PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/cell_boundaries/.zgroup'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/cell_boundaries/shapes.parquet'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/cell_boundaries/.zattrs')]
HE patches: [PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/he_patches/.zgroup'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/he_patches/shapes.parquet'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/shapes/he_patches/.zattrs')]
Tables: [PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/tables/table_nuclei'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/tables/features_phikonv2'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/tables/table_cells'), PosixPath('/content/sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/tables/.zgroup'), PosixPath('/cont

In [ ]:
import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        # Limiting files shown per directory to keep output readable if it's huge
        for f in files[:10]:
            print(f'{subindent}{f}')
        if len(files) > 10:
            print(f'{subindent}... ({len(files) - 10} more files)')

# Run it on the extracted directory
list_files('/content/sdata_lymph_node_s0')

Streaming output truncated to the last 5000 lines.
                            13
                            5
                            3
                            ... (6 more files)
                        2/
                            12
                            9
                            10
                            1
                            15
                            14
                            7
                            13
                            5
                            3
                            ... (6 more files)
                        22/
                            12
                            9
                            10
                            1
                            15
                            14
                            7
                            13
                            5
                            3
                            ... (6 more files)
                obs/
                    .zgroup
   

In [ ]:
!pip install mygene

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00


In [ ]:
import zarr
import pandas as pd
import anndata as ad
import numpy as np
import scipy.sparse as sp
from mygene import MyGeneInfo

mg = MyGeneInfo()


root_path = "sdata_lymph_node_s0/sdata_lymph_node_s0.zarr/tables/table_cells"
store = zarr.open(root_path, mode='r')

gene_ids = store["var/gene_ids"][:]

res = mg.querymany(gene_ids, scopes="ensembl.gene", fields="symbol", species="human", returnall=True)

gene_names = [r['symbol'] if "symbol" in r else r["query"] for r in res['out']]

n_cells = store["obsm/X_pca"].shape[0]
n_genes = len(gene_names)

data = store['X/data'][:]
indices = store['X/indices'][:]
indptr = store['X/indptr'][:]

full_X = sp.csr_matrix((data, indices, indptr), shape=(n_cells, n_genes))



rng = np.random.default_rng()
idx = rng.choice(n_cells, size=50_000, replace=False)

idx.sort()


print(f"Extracting first {n_cells} cells...")

spatial = store['obsm/spatial'][idx]
pca = store['obsm/X_pca'][idx]
cell_ids = store['obs/cell_id'][idx]
X_data = full_X[idx]

if 'final_label/codes' in store['obs']:
    labels = store['obs/final_label/codes'][idx]
    categories = store['obs/final_label/categories'][:]
    final_labels = [categories[i] for i in labels]
else:
    final_labels = store['obs/final_label'][idx]

print("Constructing pilot AnnData...")
adata_pilot = ad.AnnData(
    X=X_data,
    obs=pd.DataFrame({
        'cell_id': cell_ids,
        'final_label': final_labels
    }),
    var=pd.DataFrame(index=gene_names),
    obsm={
        'spatial': spatial,
        'X_pca': pca
    }
)

adata_pilot = adata_pilot[adata_pilot.obs['final_label'] != 'Less10'].copy()
adata_pilot.write_h5ad("lymph_node_50k_random3.h5ad")
print(f"Success! Created a {adata_pilot.n_obs} cell pilot file.")

INFO:biothings.client:querying 1-1000 ...


Opening Zarr store lazily...


INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-4624 ...
INFO:biothings.client:Finished.


['A2ML1', 'AAMP', 'AAR2', 'AARSD1', 'ABAT', 'ABCA1', 'ABCA3', 'ABCA4', 'ABCB1', 'ABCB4', 'ABCB6', 'ABCC1', 'ABCC12', 'ABCC2', 'ABCC3', 'ABCC4', 'ABCC6', 'ABCC8', 'ABCC9', 'ABCD1', 'ABCD3', 'ABCD4', 'ABCF3', 'ABCG1', 'ABCG2', 'ABHD11', 'ABHD6', 'ABI3BP', 'ABL1', 'ABL2', 'ABO', 'ABTB1', 'ACAA2', 'ACACA', 'ACACB', 'ACAP1', 'ACAP2', 'ACAT1', 'ACE', 'ACE2', 'ACHE', 'ACKR3', 'ACLY', 'ACOD1', 'ACP5', 'ACP3', 'ACRBP', 'ACRV1', 'ACSL4', 'ACTBL2', 'ACTL6A', 'ACTL7A', 'ACTN1', 'ACTN2', 'ACTN4', 'ACVR1', 'ACVR1B', 'ACVR2A', 'ACVRL1', 'ACYP2', 'ADA', 'ADAM10', 'ADAM12', 'ADAM17', 'ADAM28', 'ADAM33', 'ADAM8', 'ADAM9', 'ADAMDEC1', 'ADAMTS1', 'ADAMTS4', 'ADAMTS5', 'ADAR', 'ADCY2', 'ADCYAP1', 'ADD1', 'ADGRA1', 'ADGRA2', 'ADGRA3', 'ADGRB1', 'ADGRD1', 'ADGRE2', 'ADGRE5', 'ADGRL4', 'ADIPOQ', 'ADIPOR1', 'ADIPOR2', 'ADM', 'ADNP', 'ADORA1', 'ADORA2A', 'ADORA2B', 'ADORA3', 'ADRA1A', 'ADRA1B', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRA2C', 'ADRB1', 'ADRB2', 'ADRB3', 'ADTRP', 'AEBP1', 'AFAP1L2', 'AFF1', 'AFF2', 'AFG1L

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Success! Created a 49717 cell pilot file.
